# SQD Diagnostics Dashboard — Colab Edition (N2 validation run)

Live convergence diagnostics for `qiskit-addon-sqd`'s self-consistent
configuration recovery loop. Fully self-contained — no separate package
install needed, just run the cells top to bottom.

This version runs against a genuine N2 active-space problem (8 orbitals,
10 electrons, STO-3G, 2 frozen core orbitals) using a real LUCJ ansatz built
from CCSD amplitudes (the same construction IBM's own SQD tutorials use),
sampled via exact classical simulation with added hardware-like bit-flip
noise — not a trivial toy system, so the convergence plots show genuine
iteration-to-iteration dynamics instead of a flat line.

## 1. Install dependencies

In [1]:
!pip install -q qiskit-addon-sqd pyscf ffsim plotly numpy

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

## 2. The diagnostics class\n\nUses a `clear_output()` + redraw pattern rather than Plotly's `FigureWidget`, since `FigureWidget` depends on `anywidget` (as of Plotly 6.0+) which is currently broken specifically in Google Colab ([plotly/plotly.py#5027](https://github.com/plotly/plotly.py/issues/5027)). This approach works reliably across Colab, Jupyter, and VS Code notebooks.

In [2]:
"""
Live diagnostics dashboard for qiskit-addon-sqd's self-consistent
configuration recovery loop.

Usage (inside a Jupyter/JupyterLab/Colab notebook):

    from qiskit_sqd_dashboard import SQDDiagnostics
    from qiskit_addon_sqd.fermion import diagonalize_fermionic_hamiltonian

    diag = SQDDiagnostics()
    diag.display()  # renders the live figure; subsequent cells update it in place

    result = diagonalize_fermionic_hamiltonian(
        hcore, eri, bit_array, samples_per_batch=..., norb=norb, nelec=nelec,
        callback=diag.callback,
    )

The callback signature matches qiskit_addon_sqd.fermion's
`callback: Callable[[list[SCIResult]], None]` parameter exactly, so no
adapter code is needed on the user's side.
"""
from __future__ import annotations

import numpy as np


class SQDDiagnostics:
    """
    Tracks per-iteration SQD diagnostics and renders them as a live-updating
    Plotly figure inside a notebook.

    Uses a clear_output()+redraw pattern rather than go.FigureWidget, since
    FigureWidget (which depends on anywidget as of Plotly 6.0+) is broken in
    Google Colab specifically (see plotly/plotly.py#5027) — this approach
    works reliably across Jupyter, JupyterLab, Colab, and VS Code notebooks.

    Tracked per iteration (aggregated across all batches in that iteration):
        - best energy found (minimum across batches)
        - energy spread across batches (min/max), to show batch variance
        - subspace dimension per batch (len(ci_strs_a) * len(ci_strs_b))
        - orbital occupancy convergence: max abs change vs previous iteration
    """

    def __init__(self):
        self.iterations: list[int] = []
        self.best_energy: list[float] = []
        self.energy_min: list[float] = []
        self.energy_max: list[float] = []
        self.subspace_dims: list[list[int]] = []  # one list of dims per iteration (per batch)
        self.occupancy_deltas: list[float | None] = []  # None on first iteration (nothing to compare to)

        self._prev_occupancies = None
        self._displaying = False

    def callback(self, results) -> None:
        """
        Matches qiskit_addon_sqd.fermion's callback signature:
        Callable[[list[SCIResult]], None]. Pass this method directly as the
        `callback=` argument to diagonalize_fermionic_hamiltonian.
        """
        iteration = len(self.iterations)
        energies = [r.energy for r in results]
        best_idx = int(np.argmin(energies))
        best_result = results[best_idx]

        dims = [
            len(r.sci_state.ci_strs_a) * len(r.sci_state.ci_strs_b) for r in results
        ]

        occ_a, occ_b = best_result.orbital_occupancies
        current_occ = np.concatenate([occ_a, occ_b])
        if self._prev_occupancies is None:
            occ_delta = None
        else:
            occ_delta = float(np.max(np.abs(current_occ - self._prev_occupancies)))
        self._prev_occupancies = current_occ

        self.iterations.append(iteration)
        self.best_energy.append(float(energies[best_idx]))
        self.energy_min.append(float(min(energies)))
        self.energy_max.append(float(max(energies)))
        self.subspace_dims.append(dims)
        self.occupancy_deltas.append(occ_delta)

        if self._displaying:
            self._redraw()

    def display(self):
        """
        Start live display. Call this before starting the SQD run; the
        figure will redraw itself after every callback invocation.
        """
        self._displaying = True
        self._redraw()

    def _build_figure(self):
        import plotly.graph_objects as go
        from plotly.subplots import make_subplots

        fig = make_subplots(
            rows=3, cols=1,
            subplot_titles=(
                "Energy convergence", "Subspace dimension per batch", "Max orbital-occupancy change"
            ),
            vertical_spacing=0.12,
        )
        fig.add_trace(
            go.Scatter(x=self.iterations, y=self.best_energy, mode="lines+markers", name="best energy"),
            row=1, col=1,
        )
        fig.add_trace(
            go.Scatter(x=self.iterations, y=self.energy_min, mode="lines", name="energy min", line=dict(dash="dot")),
            row=1, col=1,
        )
        fig.add_trace(
            go.Scatter(x=self.iterations, y=self.energy_max, mode="lines", name="energy max", line=dict(dash="dot")),
            row=1, col=1,
        )

        batch_x, batch_y = [], []
        for it, dims in zip(self.iterations, self.subspace_dims):
            batch_x.extend([it] * len(dims))
            batch_y.extend(dims)
        fig.add_trace(
            go.Scatter(x=batch_x, y=batch_y, mode="markers", name="subspace dim (per batch)"),
            row=2, col=1,
        )

        occ_x = [it for it, d in zip(self.iterations, self.occupancy_deltas) if d is not None]
        occ_y = [d for d in self.occupancy_deltas if d is not None]
        fig.add_trace(
            go.Scatter(x=occ_x, y=occ_y, mode="lines+markers", name="max |occupancy change|"),
            row=3, col=1,
        )

        fig.update_layout(height=700, showlegend=True, margin=dict(t=60, b=40))
        fig.update_xaxes(title_text="iteration", row=3, col=1)
        fig.update_yaxes(title_text="energy (Ha)", row=1, col=1)
        fig.update_yaxes(title_text="dimension", row=2, col=1)
        fig.update_yaxes(title_text="\u0394 occupancy", row=3, col=1)
        return fig

    def _redraw(self):
        from IPython.display import clear_output
        clear_output(wait=True)
        self._build_figure().show()

    def summary(self) -> str:
        """A plain-text summary, useful outside a notebook or for logging."""
        lines = [f"SQD run: {len(self.iterations)} iterations"]
        if self.best_energy:
            lines.append(f"Final best energy: {self.best_energy[-1]:.8f} Ha")
            lines.append(f"Final subspace dims (per batch): {self.subspace_dims[-1]}")
            if self.occupancy_deltas[-1] is not None:
                lines.append(f"Final max |occupancy change|: {self.occupancy_deltas[-1]:.2e}")
        return "\n".join(lines)


## 3. Build a real N2 active-space Hamiltonian

In [3]:
from pyscf import gto, scf, mcscf, ao2mo, cc

mol = gto.M(atom="N 0 0 0; N 0 0 1.1", basis="sto-3g", symmetry="Dooh")
mf = scf.RHF(mol).run()

n_frozen = 2  # freeze the two core 1s-like orbitals
active_space = range(n_frozen, mol.nao_nr())
norb = len(active_space)
nelec = tuple(n - n_frozen for n in mol.nelec)
print(f"norb={norb}, nelec={nelec}, HF energy={mf.e_tot:.6f} Ha")

cas = mcscf.CASCI(mf, norb, nelec)
mo = cas.sort_mo(active_space, base=0)
hcore, ecore = cas.get_h1eff(mo)
eri = ao2mo.restore(1, cas.get_h2eff(mo), norb)
print(f"core energy: {ecore:.6f} Ha")


WARN: Unable to to identify input symmetry using original axes.
Different symmetry axes will be used.



converged SCF energy = -107.496500511798


norb=8, nelec=(5, 5), HF energy=-107.496501 Ha


core energy: -76.434320 Ha


## 4. Build a real LUCJ ansatz from CCSD amplitudes\n\nThis mirrors the construction used in IBM's own SQD tutorials: run CCSD to get t1/t2 amplitudes, then build a Local Unitary Cluster Jastrow (LUCJ) ansatz from them.

In [4]:
import ffsim

mycc = cc.CCSD(mf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space])
mycc.kernel()
print(f"CCSD energy: {mycc.e_tot:.6f} Ha")

n_reps = 2
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(mycc.t2, t1=mycc.t1, n_reps=n_reps)
print(f"LUCJ ansatz built with n_reps={n_reps}")

E(CCSD) = -107.64990335394  E_corr = -0.1534028421420651


CCSD energy: -107.649903 Ha
LUCJ ansatz built with n_reps=2


## 5. Simulate the circuit exactly and sample noisy bitstrings\n\n`ffsim` simulates this circuit's action on the Hartree-Fock reference state exactly (efficient fermionic simulation, not a full 2^n statevector). We then sample bitstrings from the resulting state and inject realistic per-bit hardware noise (2% flip rate, roughly matching real superconducting-qubit readout error) to produce genuinely noisy quantum samples for SQD to recover from.

In [5]:
import numpy as np
from qiskit.primitives import BitArray

reference_state = ffsim.hartree_fock_state(norb, nelec)
final_state = ffsim.apply_unitary(reference_state, ucj_op, norb=norb, nelec=nelec)

rng = np.random.default_rng(11)
clean_samples = ffsim.sample_state_vector(final_state, norb=norb, nelec=nelec, shots=3000, seed=rng)

n_bits = 2 * norb
noisy_samples = []
for s in clean_samples:
    bits = np.array([int(c) for c in s])
    flip_mask = rng.random(n_bits) < 0.02
    bits[flip_mask] = 1 - bits[flip_mask]
    noisy_samples.append("".join(str(b) for b in bits))

bit_array = BitArray.from_samples(noisy_samples, num_bits=n_bits)
print(f"{bit_array.num_shots} shots, {bit_array.num_bits} bits")

3000 shots, 16 bits


## 6. Attach the dashboard and run SQD\n\n`diag.callback` is passed directly as the `callback=` argument — no adapter code needed. The figure redraws after every configuration-recovery iteration.

In [6]:
from qiskit_addon_sqd.fermion import diagonalize_fermionic_hamiltonian

diag = SQDDiagnostics()
diag.display()

result = diagonalize_fermionic_hamiltonian(
    hcore, eri, bit_array,
    samples_per_batch=300,
    norb=norb, nelec=nelec,
    num_batches=3,
    max_iterations=10,
    callback=diag.callback,
    seed=11,
)

print(f"Final energy (active space): {result.energy:.6f} Ha")
print(f"Final total energy (with core): {result.energy + ecore:.6f} Ha")
print(f"For comparison, CCSD total energy: {mycc.e_tot:.6f} Ha")

Final energy (active space): -31.216605 Ha
Final total energy (with core): -107.650925 Ha
For comparison, CCSD total energy: -107.649903 Ha


## 7. Summary

In [7]:
print(diag.summary())

SQD run: 4 iterations
Final best energy: -31.21660473 Ha
Final subspace dims (per batch): [702, 702, 702]
Final max |occupancy change|: 0.00e+00
